# Data Augmentation: Adding Negation Examples

**Goal:** Augment the LexGLUE unfair_tos dataset with negated examples and save to Google Drive.

**What this notebook does:**
1. Load original LexGLUE dataset
2. Generate negated examples from risky clauses
3. Combine original + augmented data
4. Save augmented dataset to Google Drive

---

## 1. Setup & Installation

In [ ]:
# Install required libraries
!pip install -q datasets

In [ ]:
# Import libraries
import numpy as np
import re
from datasets import load_dataset, Dataset, DatasetDict

print("✓ Libraries imported successfully!")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for augmented data
!mkdir -p "/content/drive/MyDrive/Claussifier/data"

print("✓ Google Drive mounted!")

## 3. Load Original Dataset

In [ ]:
print("Loading LexGLUE unfair_tos dataset...")
dataset = load_dataset("lex_glue", "unfair_tos")

# Get label names
label_names = dataset['train'].features['labels'].feature.names

print(f"\n✓ Dataset loaded successfully!")
print(f"  - Train: {len(dataset['train'])} examples")
print(f"  - Validation: {len(dataset['validation'])} examples")
print(f"  - Test: {len(dataset['test'])} examples")
print(f"\nRisk Categories: {label_names}")

## 4. Define Negation Functions

In [ ]:
def has_existing_negation(clause):
    """
    Check if clause already contains negation words to avoid double negatives.
    """
    clause_lower = clause.lower()
    negation_words = [
        r'\bnot\b', r'\bno\b', r'\bnever\b', r'\bneither\b', 
        r'\bnor\b', r'\bnone\b', r'\bnothing\b', r'\bnobody\b',
        r'\bcannot\b', r"\bcan't\b", r"\bwon't\b", r"\bshouldn't\b",
        r"\bwouldn't\b", r"\bdidn't\b", r"\bdoesn't\b", r"\bdon't\b",
        r"\bisn't\b", r"\baren't\b", r"\bwasn't\b", r"\bweren't\b"
    ]
    
    for pattern in negation_words:
        if re.search(pattern, clause_lower):
            return True
    return False

def negate_clause(clause):
    """
    Apply negation to a risky clause.
    Returns (negated_clause, pattern_used) or (None, None).
    """
    # Skip clauses with existing negations
    if has_existing_negation(clause):
        return None, None
    
    clause_lower = clause.lower()
    
    # Pattern 1: Modal verb "may"
    if ' may ' in clause_lower:
        negated = re.sub(r'\bmay\b', 'may not', clause, count=1, flags=re.IGNORECASE)
        return negated, "may → may not"
    
    # Pattern 2: Modal verb "can"
    elif ' can ' in clause_lower:
        negated = re.sub(r'\bcan\b', 'cannot', clause, count=1, flags=re.IGNORECASE)
        return negated, "can → cannot"
    
    # Pattern 3: Modal verb "will"
    elif ' will ' in clause_lower:
        negated = re.sub(r'\bwill\b', 'will not', clause, count=1, flags=re.IGNORECASE)
        return negated, "will → will not"
    
    # Pattern 4: Modal verb "shall"
    elif ' shall ' in clause_lower:
        negated = re.sub(r'\bshall\b', 'shall not', clause, count=1, flags=re.IGNORECASE)
        return negated, "shall → shall not"
    
    # Pattern 5: "reserve the right"
    elif 'reserve the right' in clause_lower:
        negated = re.sub(r'reserve the right', 'do not reserve the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "reserve the right → do not reserve the right"
    
    # Pattern 6: "has the right"
    elif 'has the right' in clause_lower:
        negated = re.sub(r'has the right', 'does not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "has the right → does not have the right"
    
    # Pattern 7: "have the right"
    elif 'have the right' in clause_lower:
        negated = re.sub(r'have the right', 'do not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "have the right → do not have the right"
    
    # Pattern 8: "is/are entitled to"
    elif 'is entitled to' in clause_lower or 'are entitled to' in clause_lower:
        negated = re.sub(r'is entitled to', 'is not entitled to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'are entitled to', 'are not entitled to', negated, count=1, flags=re.IGNORECASE)
        return negated, "entitled to → not entitled to"
    
    # Pattern 9: "agree to" / "agrees to"
    elif 'agree to' in clause_lower or 'agrees to' in clause_lower:
        negated = re.sub(r'agree to', 'do not agree to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'agrees to', 'does not agree to', negated, count=1, flags=re.IGNORECASE)
        return negated, "agree to → do not agree to"
    
    return None, None

print("✓ Negation functions defined")

## 5. Augmentation Function

In [ ]:
def augment_dataset(dataset_split, augmentation_ratio=0.5):
    """
    Augment dataset with negated examples.
    
    Args:
        dataset_split: HuggingFace dataset split
        augmentation_ratio: Ratio of augmented examples to add (0.5 = 50%)
    
    Returns:
        Augmented dataset
    """
    original_examples = []
    augmented_examples = []
    
    stats = {
        'total_risky': 0,
        'total_negated': 0,
        'skipped_existing_negation': 0,
        'skipped_no_pattern': 0
    }
    
    # Process each example
    for example in dataset_split:
        # Keep original example
        original_examples.append({
            'text': example['text'],
            'labels': example['labels']
        })
        
        # Only augment risky clauses
        if len(example['labels']) > 0:
            stats['total_risky'] += 1
            
            # Try to negate
            negated_text, pattern = negate_clause(example['text'])
            
            if negated_text:
                # Randomly decide whether to add this augmented example
                if np.random.random() < augmentation_ratio:
                    augmented_examples.append({
                        'text': negated_text,
                        'labels': []  # Negated = safe (no risks)
                    })
                    stats['total_negated'] += 1
            elif has_existing_negation(example['text']):
                stats['skipped_existing_negation'] += 1
            else:
                stats['skipped_no_pattern'] += 1
    
    # Combine original + augmented
    combined_examples = original_examples + augmented_examples
    
    # Shuffle
    np.random.shuffle(combined_examples)
    
    # Convert to HuggingFace dataset
    augmented_dataset = Dataset.from_dict({
        'text': [ex['text'] for ex in combined_examples],
        'labels': [ex['labels'] for ex in combined_examples]
    })
    
    return augmented_dataset, stats

print("✓ Augmentation function defined")

## 6. Generate Augmented Dataset

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Augmentation ratio (0.5 = add 50% negated examples)
AUGMENTATION_RATIO = 0.5

print("=" * 80)
print("AUGMENTING DATASET")
print("=" * 80)
print(f"Augmentation ratio: {AUGMENTATION_RATIO * 100:.0f}%\n")

# Augment train set
print("Augmenting training set...")
augmented_train, train_stats = augment_dataset(dataset['train'], AUGMENTATION_RATIO)
print(f"  Original: {len(dataset['train'])}")
print(f"  Risky clauses: {train_stats['total_risky']}")
print(f"  Negated examples added: {train_stats['total_negated']}")
print(f"  Skipped (existing negation): {train_stats['skipped_existing_negation']}")
print(f"  Skipped (no pattern): {train_stats['skipped_no_pattern']}")
print(f"  ✓ Total: {len(augmented_train)}\n")

# Augment validation set
print("Augmenting validation set...")
augmented_val, val_stats = augment_dataset(dataset['validation'], AUGMENTATION_RATIO)
print(f"  Original: {len(dataset['validation'])}")
print(f"  Negated examples added: {val_stats['total_negated']}")
print(f"  ✓ Total: {len(augmented_val)}\n")

# Augment test set
print("Augmenting test set...")
augmented_test, test_stats = augment_dataset(dataset['test'], AUGMENTATION_RATIO)
print(f"  Original: {len(dataset['test'])}")
print(f"  Negated examples added: {test_stats['total_negated']}")
print(f"  ✓ Total: {len(augmented_test)}\n")

# Create augmented dataset dict
augmented_dataset = DatasetDict({
    'train': augmented_train,
    'validation': augmented_val,
    'test': augmented_test
})

print("=" * 80)
print("✓ Augmentation complete!")
print("=" * 80)

## 7. Save Augmented Dataset to Google Drive

In [ ]:
# Save path
SAVE_PATH = "/content/drive/MyDrive/Claussifier/data/unfair_tos_negation_augmented"

print(f"Saving augmented dataset to Google Drive...")
print(f"Path: {SAVE_PATH}")

augmented_dataset.save_to_disk(SAVE_PATH)

print("\n" + "=" * 80)
print("✓ Dataset saved successfully!")
print("=" * 80)
print(f"\nLocation: {SAVE_PATH}")
print(f"\nDataset splits:")
print(f"  - train: {len(augmented_dataset['train'])} examples")
print(f"  - validation: {len(augmented_dataset['validation'])} examples")
print(f"  - test: {len(augmented_dataset['test'])} examples")

## 8. Verify Saved Dataset (Optional)

In [ ]:
# Load the saved dataset to verify
from datasets import load_from_disk

print("Verifying saved dataset...")
loaded_dataset = load_from_disk(SAVE_PATH)

print(f"\n✓ Dataset loaded successfully!")
print(f"  - train: {len(loaded_dataset['train'])} examples")
print(f"  - validation: {len(loaded_dataset['validation'])} examples")
print(f"  - test: {len(loaded_dataset['test'])} examples")

# Show sample examples
print(f"\nSample examples from augmented training set:")
print("=" * 80)
for i in range(3):
    example = loaded_dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"  Text: {example['text'][:150]}...")
    print(f"  Labels: {example['labels']}")
    if len(example['labels']) == 0:
        print(f"  → This is likely a NEGATED example (safe clause)")
    else:
        print(f"  → This is a RISKY clause")


### 📁 Dataset Location:

```
/content/drive/MyDrive/Claussifier/data/unfair_tos_negation_augmented/
```
